# Transformer vs LSTM 訓練 (Multi30k 独→英)

In [ ]:
# リポジトリを Colab に clone
!git clone https://github.com/mukai112358/transformer.git
%cd transformer

In [ ]:
!pip install -q sacrebleu

In [ ]:
import os, sys, pickle, time
import torch

sys.path.insert(0, os.getcwd())
os.makedirs('results', exist_ok=True)

from src.data import get_dataloaders
from src.transformer import Transformer
from src.lstm_baseline import LSTMSeq2Seq
from src.train import run_training
from src.evaluate import translate_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
loaders = get_dataloaders(batch_size=64, num_workers=2)
src_vocab = loaders['src_vocab']
tgt_vocab = loaders['tgt_vocab']
print('src vocab:', len(src_vocab), 'tgt vocab:', len(tgt_vocab))
print('train batches:', len(loaders['train_loader']))

In [ ]:
# 共通ハイパラ
EPOCHS = 20
LR = 5e-4
D_MODEL = 256
NUM_HEADS = 8
D_FF = 1024
NUM_LAYERS = 3
MAX_LEN = 50
DROPOUT = 0.1

## Transformer 訓練

In [ ]:
torch.manual_seed(42)
transformer = Transformer(
    src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab),
    d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF,
    num_layers=NUM_LAYERS, max_len=MAX_LEN, dropout=DROPOUT,
)
n_params_t = sum(p.numel() for p in transformer.parameters())
print(f'Transformer params: {n_params_t:,}')

t0 = time.time()
history_t = run_training(transformer, loaders['train_loader'], loaders['val_loader'],
                         epochs=EPOCHS, lr=LR, device=device)
history_t['n_params'] = n_params_t
history_t['total_time'] = time.time() - t0

torch.save(transformer.state_dict(), 'results/transformer.pt')
with open('results/history_transformer.pkl', 'wb') as f:
    pickle.dump(history_t, f)
print(f'total time: {history_t["total_time"]:.1f}s')

## LSTM 訓練

In [ ]:
torch.manual_seed(42)
lstm = LSTMSeq2Seq(
    src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab),
    embed_dim=D_MODEL, hidden_dim=320, num_layers=NUM_LAYERS, dropout=DROPOUT,  # Transformer 10.6M に揃える
)
n_params_l = sum(p.numel() for p in lstm.parameters())
print(f'LSTM params: {n_params_l:,}')

t0 = time.time()
history_l = run_training(lstm, loaders['train_loader'], loaders['val_loader'],
                         epochs=EPOCHS, lr=LR, device=device)
history_l['n_params'] = n_params_l
history_l['total_time'] = time.time() - t0

torch.save(lstm.state_dict(), 'results/lstm.pt')
with open('results/history_lstm.pkl', 'wb') as f:
    pickle.dump(history_l, f)
print(f'total time: {history_l["total_time"]:.1f}s')

## 訳例の確認と保存

In [ ]:
hyps_t, refs, src_lens = translate_dataset(transformer, loaders['test_loader'], tgt_vocab, device)
hyps_l, _, _ = translate_dataset(lstm, loaders['test_loader'], tgt_vocab, device)

for i in range(5):
    print(f'--- sample {i} ---')
    print('REF :', refs[i])
    print('TRA :', hyps_t[i])
    print('LSTM:', hyps_l[i])
    print()

with open('results/translations_test.pkl', 'wb') as f:
    pickle.dump({'refs': refs, 'transformer': hyps_t, 'lstm': hyps_l, 'src_lens': src_lens}, f)
print('saved.')